In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def coalesce_columns_vectorized(df: pd.DataFrame, base_name: str) -> pd.Series:
    """
    Combines values from columns representing the same field across different sources
    (Settlement, API, Mongo) into one canonical column.
    Priority order: Settlement > API > Mongo.

    Args:
        df (pd.DataFrame): The merged DataFrame containing source-specific columns.
        base_name (str): The base name of the column (e.g., 'mrch_id').

    Returns:
        pd.Series: A Series with the coalesced values for the specified base_name.
    """
    # Construct the expected column names with suffixes for each source
    set_col = f"{base_name}_set"
    api_col = f"{base_name}_api"
    mongo_col = base_name

    # Initialize the output Series with all NA values
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    print("Initial empty output Series:")
    display(out)

    # 1) Fill from Settlement if the settlement column exists
    if set_col in df.columns:
        out = df[set_col].where(df[set_col].notna(), out)
    print("After filling from Settlement:")
    display(out)

    # 2) Fill from API where Settlement was missing
    if api_col in df.columns:
        out = out.where(out.notna(), df[api_col])

    print("After filling from API:")
    display(out)

    # 3) Fill from Mongo when both Settlement & API were missing
    if mongo_col in df.columns:
        out = out.where(out.notna(), df[mongo_col])
    
    print("After filling from Mongo:")
    display(out)

    return out

In [ ]:
# 1. Define a sample 'merged' DataFrame
# This simulates the result of merging data from Settlement, API, and Mongo.
data = {
    'tran_id_raw': ['T1', 'T2', 'T3', 'T4', 'T5'],  # Transaction ID
    'mrch_id_set': ['M101', np.nan, 'M103', 'M104', np.nan], # Merchant ID from Settlement
    'mrch_id_api': [np.nan, 'M202', 'M103', np.nan, np.nan], # Merchant ID from API
    'mrch_id': [np.nan, np.nan, np.nan, 'M104', 'M105'] # Merchant ID from Mongo
}
merged_df = pd.DataFrame(data)

print("--- Original Merged DataFrame ---")
print(merged_df)
print("\n")

# 2. Call the function to coalesce the 'mrch_id' variants
canonical_mrch_id = coalesce_columns_vectorized(merged_df, 'mrch_id')

print("--- Canonical 'mrch_id' Series ---")
print(canonical_mrch_id)
print("\n")

# 3. To demonstrate how this integrates into a full canonical DataFrame:
final_canonical_df_snippet = pd.DataFrame({
    'tran_id_raw': merged_df['tran_id_raw'],
    'mrch_id': canonical_mrch_id
})
print("--- Resulting Canonical DataFrame Snippet (for 'mrch_id') ---")
print(final_canonical_df_snippet)
print("\n")
